In [2]:
!pip install transformers peft bitsandbytes accelerate datasets -q
print("✅ Libraries installed!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
✅ Libraries installed!


In [3]:
from google.colab import files
uploaded = files.upload()

Saving clean_dataset.json to clean_dataset.json


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config — this is what makes 7B fit on free GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Mistral-7B in 4-bit (this takes a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"\n✅ Model loaded!")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


✅ Model loaded!
Memory footprint: 4.01 GB


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training in 4-bit
model = prepare_model_for_kbit_training(model)

# LoRA configuration — this is what we'll experiment with
lora_config = LoraConfig(
    r=16,                    # rank — controls how many trainable params
    lora_alpha=32,           # scaling factor
    target_modules=[         # which layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap model with LoRA adapters
model = get_peft_model(model, lora_config)

# Show how few parameters we're actually training
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [6]:
import json
from datasets import Dataset

# Load your clean dataset
with open("clean_dataset.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples")

# Convert to HuggingFace Dataset format
hf_dataset = Dataset.from_list(data)

def tokenize_function(examples):
    """Tokenize the formatted instruction text."""
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

print("Tokenizing dataset...")
tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=hf_dataset.column_names
)

# Split into train/val
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"\n✅ Train examples: {len(train_dataset)}")
print(f"✅ Val examples: {len(val_dataset)}")
print(f"\nSample tokenized input_ids length: {len(train_dataset[0]['input_ids'])}")

Loaded 5000 examples
Tokenizing dataset...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


✅ Train examples: 4500
✅ Val examples: 500

Sample tokenized input_ids length: 512


In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training arguments
training_args = TrainingArguments(
    output_dir="./mistral-code-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch size = 8
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,                          # mixed precision
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",           # memory efficient optimizer
    report_to="none",
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("🚀 Starting QLoRA fine-tuning of Mistral-7B...")
print("This will take 30-45 minutes on free T4 GPU\n")

trainer.train()

print("\n🎉 Training complete!")

🚀 Starting QLoRA fine-tuning of Mistral-7B...
This will take 30-45 minutes on free T4 GPU



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.589437,0.578355
200,0.564816,0.562695
300,0.601084,0.554059
400,0.568691,0.547049
500,0.550314,0.542347
563,0.529769,0.542020


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



🎉 Training complete!


In [8]:
# Save the LoRA adapter (small file, only the trained layers)
model.save_pretrained("mistral-code-lora-adapter")
tokenizer.save_pretrained("mistral-code-lora-adapter")
print("✅ LoRA adapter saved!")

# Check the size — it's tiny compared to the full model
import os
total_size = 0
for path, dirs, files in os.walk("mistral-code-lora-adapter"):
    for f in files:
        total_size += os.path.getsize(os.path.join(path, f))
print(f"Adapter size: {total_size / 1e6:.1f} MB")
print("(vs 14GB for the full model — that's the power of LoRA!)")

✅ LoRA adapter saved!
Adapter size: 58.1 MB
(vs 14GB for the full model — that's the power of LoRA!)


In [9]:
def generate_code(instruction, max_new_tokens=200):
    """Generate code from an instruction using the fine-tuned model."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = full_output.split("### Response:\n")[-1]
    return response


# Test with coding tasks
test_instructions = [
    "Write a Python function to check if a number is prime.",
    "Create a function that reverses a string.",
    "Write a Python function to find the maximum element in a list.",
]

for instruction in test_instructions:
    print("="*70)
    print(f"INSTRUCTION: {instruction}")
    print("="*70)
    print(generate_code(instruction))
    print("\n")

INSTRUCTION: Write a Python function to check if a number is prime.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


def is_prime(n):
    if n == 2:
        return True
    if n <= 1 or n % 2 == 0:
        return False
    for i in range(3, int(n ** 0.5) + 1, 2):
        if n % i == 0:
            return False
    return True

if __name__ == '__main__':
    print(is_prime(17)) # Output: True
    print(is_prime(16)) # Output: False
    print(is_prime(0)) # Output: False
    print(is_prime(-1)) # Output: False

# This code checks if a given number is prime or not.
# It uses the Euclidean algorithm to check if the number is divisible by any other number.
# If the number is not divisible by


INSTRUCTION: Create a function that reverses a string.
def reverse_string(string):
    return string[::-1]

if __name__ == '__main__':
    print(reverse_string("hello world")) # prints "dlrow olleh"

# Output:
# dlrow olleh

# Note: The string is reversed in place.
# The output is the same as the input.

# Output:
# dlrow olleh

# Note: The string is reversed in place.
# The output is the same as the input.

# Outpu

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
# Zip the adapter
shutil.make_archive("/content/mistral-code-lora-adapter", "zip", "mistral-code-lora-adapter")
# Copy to Drive
shutil.copy("/content/mistral-code-lora-adapter.zip", "/content/drive/MyDrive/mistral-code-lora-adapter.zip")
print("✅ Adapter saved to Google Drive!")

# Also download to your computer
files.download("/content/mistral-code-lora-adapter.zip")
print("✅ Downloading to your computer...")

Mounted at /content/drive
✅ Adapter saved to Google Drive!


AttributeError: 'list' object has no attribute 'download'

In [11]:
from google.colab import files as colab_files
colab_files.download("/content/mistral-code-lora-adapter.zip")
print("✅ Downloading to your computer...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading to your computer...


In [12]:
# Load a FRESH base Mistral (no fine-tuning) to compare against
from transformers import AutoModelForCausalLM

print("Loading base Mistral-7B for comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto",
)
print("✅ Base model loaded!")


def generate_from_model(model_to_use, instruction, max_new_tokens=150):
    """Generate code from a given model."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)
    with torch.no_grad():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full.split("### Response:\n")[-1]


print("✅ Ready for comparison!")

Loading base Mistral-7B for comparison...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Base model loaded!
✅ Ready for comparison!


In [13]:
test_prompts = [
    "Write a Python function to check if a string is a palindrome.",
    "Create a function to calculate the factorial of a number.",
    "Write a function that returns the Fibonacci sequence up to n terms.",
]

for instruction in test_prompts:
    print("="*75)
    print(f"📝 INSTRUCTION: {instruction}")
    print("="*75)

    print("\n🔵 BASE MISTRAL (no fine-tuning):")
    print("-"*40)
    print(generate_from_model(base_model, instruction))

    print("\n🟢 YOUR FINE-TUNED MISTRAL:")
    print("-"*40)
    print(generate_from_model(model, instruction))

    print("\n\n")

📝 INSTRUCTION: Write a Python function to check if a string is a palindrome.

🔵 BASE MISTRAL (no fine-tuning):
----------------------------------------

```python
def is_palindrome(string):
    return string == string[::-1]
```

### Explanation:

This is a simple function that takes a string as input and checks if it is a palindrome.

The function returns True if the string is a palindrome and False otherwise.

The function checks if the string is equal to its reverse.

The function uses a slice notation to reverse the string.

### Test Cases:
```python
is_palindrome("racecar")
# Output: True

is_palindrome("abba")
# Output: True

is_

🟢 YOUR FINE-TUNED MISTRAL:
----------------------------------------
def is_palindrome(string): 
    return string == string[::-1] 

if __name__ == "__main__":
    string = "radar"
    print(is_palindrome(string))

Output: True

# Output:
# True

def is_palindrome(string): 
    return string == string[::-1] 

if __name__ == "__main__":
    string = "level

In [14]:
def code_quality_metrics(text):
    """Simple metrics to evaluate generated code."""
    lines = text.strip().split("\n")

    # Does it contain a function definition?
    has_function = "def " in text or "function" in text

    # Does it have a docstring?
    has_docstring = '"""' in text or "'''" in text

    # Repetition: ratio of unique lines to total lines
    non_empty = [l.strip() for l in lines if l.strip()]
    unique_ratio = len(set(non_empty)) / len(non_empty) if non_empty else 0

    return {
        "has_function": has_function,
        "has_docstring": has_docstring,
        "unique_line_ratio": round(unique_ratio, 3),
        "num_lines": len(non_empty),
    }


# Compare metrics across both models
test_prompts = [
    "Write a Python function to check if a number is even.",
    "Create a function to sort a list of numbers.",
    "Write a function to count vowels in a string.",
]

print(f"{'Metric':<22} {'Base Mistral':>15} {'Fine-Tuned':>15}")
print("="*55)

base_scores = {"has_function": 0, "has_docstring": 0, "unique_ratio": 0}
ft_scores = {"has_function": 0, "has_docstring": 0, "unique_ratio": 0}

for instruction in test_prompts:
    base_out = generate_from_model(base_model, instruction)
    ft_out = generate_from_model(model, instruction)

    bm = code_quality_metrics(base_out)
    fm = code_quality_metrics(ft_out)

    base_scores["has_function"] += bm["has_function"]
    base_scores["has_docstring"] += bm["has_docstring"]
    base_scores["unique_ratio"] += fm["unique_line_ratio"]

    ft_scores["has_function"] += fm["has_function"]
    ft_scores["has_docstring"] += fm["has_docstring"]
    ft_scores["unique_ratio"] += fm["unique_line_ratio"]

n = len(test_prompts)
print(f"{'Has function (%)':<22} {base_scores['has_function']/n*100:>14.0f}% {ft_scores['has_function']/n*100:>14.0f}%")
print(f"{'Has docstring (%)':<22} {base_scores['has_docstring']/n*100:>14.0f}% {ft_scores['has_docstring']/n*100:>14.0f}%")
print(f"{'Avg unique line ratio':<22} {base_scores['unique_ratio']/n:>15.3f} {ft_scores['unique_ratio']/n:>15.3f}")

Metric                    Base Mistral      Fine-Tuned
Has function (%)                  100%            100%
Has docstring (%)                   0%             33%
Avg unique line ratio            0.741           0.741


In [15]:
!pip install gradio -q
print("✅ Gradio installed!")

✅ Gradio installed!


In [16]:
import gradio as gr

def generate_code_demo(instruction, temperature, max_tokens):
    """Generate code from user instruction for the Gradio demo."""
    if not instruction.strip():
        return "Please enter a coding instruction."

    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            temperature=float(temperature),
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,   # reduces the repetition we saw earlier
            pad_token_id=tokenizer.eos_token_id,
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full.split("### Response:\n")[-1]
    return response


# Build the Gradio interface
demo = gr.Interface(
    fn=generate_code_demo,
    inputs=[
        gr.Textbox(
            label="Coding Instruction",
            placeholder="e.g. Write a Python function to sort a dictionary by value",
            lines=3
        ),
        gr.Slider(0.1, 1.5, value=0.7, label="Temperature (creativity)"),
        gr.Slider(50, 400, value=200, step=50, label="Max tokens"),
    ],
    outputs=gr.Code(label="Generated Code", language="python"),
    title="🤖 Mistral-7B Code Generator (QLoRA Fine-Tuned)",
    description="Fine-tuned Mistral-7B on CodeAlpaca using QLoRA. Type a coding task and watch it generate Python code!",
    examples=[
        ["Write a Python function to check if a number is prime.", 0.7, 200],
        ["Create a function that merges two sorted lists.", 0.7, 200],
        ["Write a function to find duplicates in a list.", 0.7, 200],
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0d8876afa43bbc962.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
